In [ ]:
# In[1]:

import pandas as pd

# Load CSVs (they are in the working directory)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_err = pd.read_csv("error_logs.csv")

# Helper to summarize a generic dataframe with timestamp column and name column
def summarize_df(df, name_col):
    # Ensure timestamp exists and compute min/max safely
    if 'timestamp' in df.columns and not df['timestamp'].empty:
        ts_min = int(df['timestamp'].min())
        ts_max = int(df['timestamp'].max())
        ts_min_iso = pd.to_datetime(ts_min, unit='s', utc=True).isoformat()
        ts_max_iso = pd.to_datetime(ts_max, unit='s', utc=True).isoformat()
    else:
        ts_min = ts_max = ts_min_iso = ts_max_iso = None

    cmdb_vals = df['cmdb_id'].dropna().unique()[:50].tolist() if 'cmdb_id' in df.columns else []
    cmdb_count = int(df['cmdb_id'].nunique()) if 'cmdb_id' in df.columns else 0
    total_rows = int(len(df))

    # Top name values (limit to 20). Handle missing name_col gracefully.
    if name_col in df.columns:
        top_names = df[name_col].fillna("<NA>").astype(str).value_counts().head(20).rename_axis(name_col).reset_index(name='count')
    else:
        top_names = pd.DataFrame(columns=[name_col, 'count'])

    return {
        "total_rows": total_rows,
        "unique_cmdb_ids_preview": cmdb_vals,
        "unique_cmdb_count": cmdb_count,
        "min_timestamp": ts_min,
        "min_timestamp_iso_utc": ts_min_iso,
        "max_timestamp": ts_max,
        "max_timestamp_iso_utc": ts_max_iso,
        "top_20_names": top_names  # pandas DataFrame (name, count)
    }

# Summaries for metric, trace, log
metric_summary = summarize_df(df_metric, "kpi_name")
trace_summary = summarize_df(df_trace, "trace_name")
log_summary = summarize_df(df_log, "log_name")

# For error_logs we need to show top 20 messages truncated to 200 chars with counts.
if 'timestamp' in df_err.columns and not df_err['timestamp'].empty:
    err_ts_min = int(df_err['timestamp'].min())
    err_ts_max = int(df_err['timestamp'].max())
    err_ts_min_iso = pd.to_datetime(err_ts_min, unit='s', utc=True).isoformat()
    err_ts_max_iso = pd.to_datetime(err_ts_max, unit='s', utc=True).isoformat()
else:
    err_ts_min = err_ts_max = err_ts_min_iso = err_ts_max_iso = None

err_cmdb_vals = df_err['cmdb_id'].dropna().unique()[:50].tolist() if 'cmdb_id' in df_err.columns else []
err_cmdb_count = int(df_err['cmdb_id'].nunique()) if 'cmdb_id' in df_err.columns else 0
err_total_rows = int(len(df_err))

# Truncate messages to 200 chars then count
if 'message' in df_err.columns:
    truncated = df_err['message'].fillna('').astype(str).str.slice(0, 200)
    top_err_msgs = truncated.value_counts().head(20).rename_axis('message').reset_index(name='count')
else:
    top_err_msgs = pd.DataFrame(columns=['message', 'count'])

error_logs_summary = {
    "total_rows": err_total_rows,
    "unique_cmdb_ids_preview": err_cmdb_vals,
    "unique_cmdb_count": err_cmdb_count,
    "min_timestamp": err_ts_min,
    "min_timestamp_iso_utc": err_ts_min_iso,
    "max_timestamp": err_ts_max,
    "max_timestamp_iso_utc": err_ts_max_iso,
    "top_20_messages_truncated_200": top_err_msgs
}

# Compose compact JSON-like summary
summary = {
    "metric.csv": metric_summary,
    "trace.csv": trace_summary,
    "log.csv": log_summary,
    "error_logs.csv": error_logs_summary
}

# Display the compact summary (DataFrames for top lists are small: <=20 rows)
summary

```
Out[1]:
```
Summary (plain English, timestamps in UTC):

- metric.csv
  - total rows: 1875
  - unique cmdb_ids (preview, 15): carts, carts-db, catalogue, catalogue-db, front-end, orders, orders-db, payment, queue-master, rabbitmq, rabbitmq-exporter, session-db, shipping, user, user-db
  - number of unique cmdb_ids: 15
  - time range: earliest 2024-01-20T03:53:00+00:00, latest 2024-01-20T04:17:00+00:00
  - top KPI names (name: count, top 7 shown): cpu: 375, socket: 375, mem: 375, diskio: 225, latency-50: 175, latency-90: 175, workload: 175

- trace.csv
  - total rows: 0 (no trace data available)
  - no cmdb_ids, no timestamps, no trace names

- log.csv
  - total rows: 454
  - unique cmdb_ids (preview, 11): carts, carts-db, catalogue, front-end, orders, payment, queue-master, shipping, user, session-db, user-db
  - number of unique cmdb_ids: 11
  - time range: earliest 2024-01-20T03:53:00+00:00, latest 2024-01-20T04:17:00+00:00
  - top log names (name: count): log.error_count: 227, log.row_count: 227

- error_logs.csv
  - total rows: 0 (no error log messages)
  - no cmdb_ids, no timestamps, no messages

If you want, I can export these top lists in a compact table or filter any of them further.

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'total_rows': 1875, 'unique_cmdb_ids_preview': ['carts', 'carts-db', 'catalogue', 'catalogue-db', 'front-end', 'orders', 'orders-db', 'payment', 'queue-master', 'rabbitmq', 'rabbitmq-exporter', 'session-db', 'shipping', 'user', 'user-db'], 'unique_cmdb_count': 15, 'min_timestamp': 1705722780, 'min_timestamp_iso_utc': '2024-01-20T03:53:00+00:00', 'max_timestamp': 1705724220, 'max_timestamp_iso_utc': '2024-01-20T04:17:00+00:00', 'top_20_names':      kpi_name  count
0         cpu    375
1      socket    375
2         mem    375
3      diskio    225
4  latency-50    175
5  latency-90    175
6    workload    175}, 'trace.csv': {'total_rows': 0, 'unique_cmdb_ids_preview': [], 'unique_cmdb_count': 0, 'min_timestamp': None, 'min_timestamp_iso_utc': None, 'max_timestamp': None, 'max_timestamp_iso_utc': None, 'top_20_names': Empty DataFrame
Columns: [trace_name, count]
Index: []}, 'log.csv': {'total_rows': 454, 'unique_cmdb_ids_preview': ['carts', 'carts-db', 'catalogue', 'front-end', 'orders', 'payment', 'queue-master', 'shipping', 'user', 'session-db', 'user-db'], 'unique_cmdb_count': 11, 'min_timestamp': 1705722780, 'min_timestamp_iso_utc': '2024-01-20T03:53:00+00:00', 'max_timestamp': 1705724220, 'max_timestamp_iso_utc': '2024-01-20T04:17:00+00:00', 'top_20_names':           log_name  count
0  log.error_count    227
1    log.row_count    227}, 'error_logs.csv': {'total_rows': 0, 'unique_cmdb_ids_preview': [], 'unique_cmdb_count': 0, 'min_timestamp': None, 'min_timestamp_iso_utc': None, 'max_timestamp': None, 'max_timestamp_iso_utc': None, 'top_20_messages_truncated_200': Empty DataFrame
Columns: [message, count]
Index: []}}```
```

In [ ]:
# In[2]:

import pandas as pd
# Reuse previously loaded df_metric and df_log

# Ensure timestamp as UTC datetimes (Unix seconds)
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-20 03:50:34", utc=True)
window_end = pd.to_datetime("2024-01-20 04:20:34", utc=True)

# ---- Metrics: global P95 and totals (computed on full series) ----
metric_baseline = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(global_p95=lambda x: float(x.quantile(0.95)), total_points='count')
    .reset_index()
)

# Filter metric rows in window
df_metric_win = df_metric[(df_metric['ts'] >= window_start) & (df_metric['ts'] <= window_end)].copy()

# Points and max in window per pair
metric_window_stats = (
    df_metric_win
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(points_in_window='count', max_value_in_window='max')
    .reset_index()
)

# For anomaly counts and earliest anomaly timestamp we need per-row comparison with global_p95
# Merge baseline p95 into window rows
if not df_metric_win.empty:
    mrows = df_metric_win.merge(metric_baseline, on=['cmdb_id','kpi_name'], how='left', validate='m:m')
    mrows['is_anom'] = (mrows['value'] >= mrows['global_p95']).astype(int)
    # earliest anomaly ts per pair
    anomaly_agg = (
        mrows[mrows['is_anom'] == 1]
        .groupby(['cmdb_id','kpi_name'], dropna=False)
        .agg(anomaly_count_in_window=('is_anom','sum'),
             earliest_anomaly_ts=('ts','min'))
        .reset_index()
    )
    # format earliest timestamp to ISO UTC strings
    if not anomaly_agg.empty:
        anomaly_agg['earliest_anomaly_timestamp_iso_utc'] = anomaly_agg['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').replace({'+0000': '+00:00'})
        anomaly_agg = anomaly_agg.drop(columns=['earliest_anomaly_ts'])
else:
    anomaly_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count_in_window','earliest_anomaly_timestamp_iso_utc'])

# Combine baseline, window stats, anomaly aggregates
metric_merged = (
    metric_baseline
    .merge(metric_window_stats, on=['cmdb_id','kpi_name'], how='left')
    .merge(anomaly_agg, on=['cmdb_id','kpi_name'], how='left')
)

# Fill NaNs for pairs with no points in window
metric_merged['points_in_window'] = metric_merged['points_in_window'].fillna(0).astype(int)
metric_merged['anomaly_count_in_window'] = metric_merged['anomaly_count_in_window'].fillna(0).astype(int)
metric_merged['max_value_in_window'] = metric_merged['max_value_in_window'].where(metric_merged['max_value_in_window'].notna(), None)
metric_merged['earliest_anomaly_timestamp_iso_utc'] = metric_merged['earliest_anomaly_timestamp_iso_utc'].where(metric_merged['earliest_anomaly_timestamp_iso_utc'].notna(), None)

# Select required columns and sort
metrics_result = (
    metric_merged[['cmdb_id','kpi_name','global_p95','total_points','points_in_window','anomaly_count_in_window','earliest_anomaly_timestamp_iso_utc','max_value_in_window']]
    .sort_values(by=['anomaly_count_in_window','max_value_in_window'], ascending=[False, False])
    .reset_index(drop=True)
)

# Limit to top 20 rows
metrics_result_top20 = metrics_result.head(20)

# Count of distinct (cmdb_id,kpi_name)
metrics_pairs_count = int(metric_baseline.shape[0])

# ---- Logs: same procedure with log_name ----
log_baseline = (
    df_log
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(global_p95=lambda x: float(x.quantile(0.95)), total_points='count')
    .reset_index()
)

df_log_win = df_log[(df_log['ts'] >= window_start) & (df_log['ts'] <= window_end)].copy()

log_window_stats = (
    df_log_win
    .groupby(['cmdb_id','log_name'], dropna=False)['value']
    .agg(points_in_window='count', max_value_in_window='max')
    .reset_index()
)

if not df_log_win.empty:
    lrows = df_log_win.merge(log_baseline, on=['cmdb_id','log_name'], how='left', validate='m:m')
    lrows['is_anom'] = (lrows['value'] >= lrows['global_p95']).astype(int)
    anomaly_agg_log = (
        lrows[lrows['is_anom'] == 1]
        .groupby(['cmdb_id','log_name'], dropna=False)
        .agg(anomaly_count_in_window=('is_anom','sum'),
             earliest_anomaly_ts=('ts','min'))
        .reset_index()
    )
    if not anomaly_agg_log.empty:
        anomaly_agg_log['earliest_anomaly_timestamp_iso_utc'] = anomaly_agg_log['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').replace({'+0000': '+00:00'})
        anomaly_agg_log = anomaly_agg_log.drop(columns=['earliest_anomaly_ts'])
else:
    anomaly_agg_log = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count_in_window','earliest_anomaly_timestamp_iso_utc'])

log_merged = (
    log_baseline
    .merge(log_window_stats, on=['cmdb_id','log_name'], how='left')
    .merge(anomaly_agg_log, on=['cmdb_id','log_name'], how='left')
)

log_merged['points_in_window'] = log_merged['points_in_window'].fillna(0).astype(int)
log_merged['anomaly_count_in_window'] = log_merged['anomaly_count_in_window'].fillna(0).astype(int)
log_merged['max_value_in_window'] = log_merged['max_value_in_window'].where(log_merged['max_value_in_window'].notna(), None)
log_merged['earliest_anomaly_timestamp_iso_utc'] = log_merged['earliest_anomaly_timestamp_iso_utc'].where(log_merged['earliest_anomaly_timestamp_iso_utc'].notna(), None)

logs_result = (
    log_merged[['cmdb_id','log_name','global_p95','total_points','points_in_window','anomaly_count_in_window','earliest_anomaly_timestamp_iso_utc','max_value_in_window']]
    .sort_values(by=['anomaly_count_in_window','max_value_in_window'], ascending=[False, False])
    .reset_index(drop=True)
)

logs_result_top20 = logs_result.head(20)
logs_pairs_count = int(log_baseline.shape[0])

# Compact outputs (DataFrames limited to <=20 rows)
metrics_result_top20, logs_result_top20, metrics_pairs_count, logs_pairs_count

```
Out[2]:
```
Summary (incident window: 2024-01-20 03:50:34 → 2024-01-20 04:20:34 UTC)

Metrics (processed 75 distinct (cmdb_id,kpi_name) pairs)
- Top anomalous pairs (sorted by anomaly_count_in_window desc then max value):
  - rabbitmq-exporter / mem — global P95 ≈ 4.616e+06, total_points 25, points_in_window 25, anomaly_count_in_window 25, earliest anomaly ~ 2024-01-20T03:53:00Z, max ≈ 4.616e+06
  - rabbitmq / socket — global P95 = 11.0, total_points 25, points_in_window 25, anomaly_count_in_window 25, earliest anomaly ~ 2024-01-20T03:53:00Z, max = 11.0
  - catalogue / socket — global P95 = 7.0, total_points 25, points_in_window 25, anomaly_count_in_window 25, earliest anomaly ~ 2024-01-20T03:53:00Z, max = 7.0
  - payment / socket — global P95 = 6.0, total_points 25, points_in_window 25, anomaly_count_in_window 25, earliest anomaly ~ 2024-01-20T03:53:00Z, max = 6.0
  - orders-db / socket — global P95 = 5.0, total_points 25, points_in_window 25, anomaly_count_in_window 25, earliest anomaly ~ 2024-01-20T03:53:00Z, max = 5.0
- Other notable metric observations: many socket and mem KPIs across services had all window points at or above their global P95 (anomaly_count_in_window = 25 for many), while some mem KPIs show only a few anomalies (e.g., catalogue mem, orders mem, shipping mem with anomaly_count_in_window = 2).

Logs (processed 22 distinct (cmdb_id,log_name) pairs)
- Top anomalous log pairs:
  - catalogue, front-end, orders, payment, queue-master, shipping, user — log.error_count: global P95 = 0.0, total_points 25 each, points_in_window 25, anomaly_count_in_window 25, earliest anomaly ~ 2024-01-20T03:53:00Z, max value 0
    - (These indicate the log.error_count series equals or exceeds its P95 (0) for all points — i.e., zero is the 95th percentile baseline.)
  - carts-db / log.row_count — global P95 = 1.0, total_points 24, points_in_window 24, anomaly_count_in_window 24, earliest anomaly ~ 2024-01-20T03:53:00Z, max = 2
  - carts / log.row_count — global P95 = 100.0, total_points 23, points_in_window 23, anomaly_count_in_window 6, earliest anomaly ~ 2024-01-20T03:55:00Z, max = 125
  - queue-master / log.row_count — global P95 = 585.0, total_points 25, points_in_window 25, anomaly_count_in_window 4, earliest anomaly ~ 2024-01-20T03:57:00Z, max = 594
  - several services (front-end, user, catalogue) show elevated log.row_count with small anomaly counts (2) and high max row counts.
- Note: many log.error_count series have global P95 = 0, so they appear as anomalies whenever value >= 0 (all points).

Counts
- Distinct metric (cmdb_id,kpi_name) pairs processed: 75
- Distinct log (cmdb_id,log_name) pairs processed: 22

If you want, I can return the compact top-20 tables shown above as CSV-like text or further filter to specific services/KPIs.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name    global_p95  total_points  points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_iso_utc  max_value_in_window
0   rabbitmq-exporter      mem  4.616192e+06            25                25                       25           2024-01-20T03:53:00+0000         4.616192e+06
1            rabbitmq   socket  1.100000e+01            25                25                       25           2024-01-20T03:53:00+0000         1.100000e+01
2           catalogue   socket  7.000000e+00            25                25                       25           2024-01-20T03:53:00+0000         7.000000e+00
3             payment   socket  6.000000e+00            25                25                       25           2024-01-20T03:53:00+0000         6.000000e+00
4           orders-db   socket  5.000000e+00            25                25                       25           2024-01-20T03:53:00+0000         5.000000e+00
5        catalogue-db   socket  4.000000e+00            25                25                       25           2024-01-20T03:53:00+0000         4.000000e+00
6        queue-master   socket  3.000000e+00            25                25                       25           2024-01-20T03:53:00+0000         3.000000e+00
7   rabbitmq-exporter   socket  1.000000e+00            25                25                       25           2024-01-20T03:53:00+0000         1.000000e+00
8        catalogue-db   diskio  0.000000e+00            25                25                       25           2024-01-20T03:53:00+0000         0.000000e+00
9              orders   diskio  0.000000e+00            25                25                       25           2024-01-20T03:53:00+0000         0.000000e+00
10           rabbitmq   diskio  0.000000e+00            25                25                       25           2024-01-20T03:53:00+0000         0.000000e+00
11               user   socket  2.000000e+01            25                25                       24           2024-01-20T03:53:00+0000         2.000000e+01
12            user-db   socket  1.000000e+01            25                25                       13           2024-01-20T03:53:00+0000         1.000000e+01
13           carts-db   socket  8.000000e+00            25                25                       10           2024-01-20T04:08:00+0000         8.000000e+00
14       catalogue-db      mem  4.310393e+08            25                25                        2           2024-01-20T03:54:00+0000         4.310997e+08
15             orders      mem  3.203468e+08            25                25                        2           2024-01-20T04:11:00+0000         3.216684e+08
16           shipping      mem  3.056361e+08            25                25                        2           2024-01-20T04:14:00+0000         3.056380e+08
17       queue-master      mem  2.725203e+08            25                25                        2           2024-01-20T04:09:00+0000         2.725502e+08
18              carts      mem  2.279152e+08            25                25                        2           2024-01-20T04:10:00+0000         2.282720e+08
19          front-end      mem  1.063129e+08            25                25                        2           2024-01-20T04:04:00+0000         1.078989e+08,          cmdb_id         log_name  global_p95  total_points  points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_iso_utc  max_value_in_window
0      catalogue  log.error_count         0.0            25                25                       25           2024-01-20T03:53:00+0000                    0
1      front-end  log.error_count         0.0            25                25                       25           2024-01-20T03:53:00+0000                    0
2         orders  log.error_count         0.0            25                25                       25           2024-01-20T03:53:00+0000                    0
3        payment  log.error_count         0.0            25                25                       25           2024-01-20T03:53:00+0000                    0
4   queue-master  log.error_count         0.0            25                25                       25           2024-01-20T03:53:00+0000                    0
5       shipping  log.error_count         0.0            25                25                       25           2024-01-20T03:53:00+0000                    0
6           user  log.error_count         0.0            25                25                       25           2024-01-20T03:53:00+0000                    0
7       carts-db    log.row_count         1.0            24                24                       24           2024-01-20T03:53:00+0000                    2
8       carts-db  log.error_count         0.0            24                24                       24           2024-01-20T03:53:00+0000                    0
9          carts  log.error_count         0.0            23                23                       23           2024-01-20T03:53:00+0000                    0
10         carts    log.row_count       100.0            23                23                        6           2024-01-20T03:55:00+0000                  125
11  queue-master    log.row_count       585.0            25                25                        4           2024-01-20T03:57:00+0000                  594
12        orders    log.row_count       130.0            25                25                        4           2024-01-20T03:57:00+0000                  132
13       payment    log.row_count       105.0            25                25                        4           2024-01-20T03:57:00+0000                  106
14      shipping    log.row_count        65.0            25                25                        4           2024-01-20T03:57:00+0000                   66
15    session-db    log.row_count         5.0             4                 4                        4           2024-01-20T03:57:00+0000                    5
16    session-db  log.error_count         0.0             4                 4                        4           2024-01-20T03:57:00+0000                    0
17     front-end    log.row_count      2047.4            25                25                        2           2024-01-20T04:10:00+0000                 2057
18          user    log.row_count       617.8            25                25                        2           2024-01-20T04:12:00+0000                  623
19     catalogue    log.row_count       166.8            25                25                        2           2024-01-20T03:54:00+0000                  171, 75, 22)```
```

In [ ]:
# In[3]:

# Fix and rerun focused metric fault-identification for selected components
import pandas as pd
target_ids = ["carts","catalogue","orders","payment","user"]

# Subset metric rows for the target components (full series kept for global P95)
df_metric_sub = df_metric[df_metric['cmdb_id'].isin(target_ids)].copy()

# 1) Global P95 and total_points using full series
metric_baseline = (
    df_metric_sub
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(global_p95=lambda x: float(x.quantile(0.95)), total_points='count')
    .reset_index()
)

# 2) Filter to incident window and compute window stats
df_metric_win_sub = df_metric_sub[(df_metric_sub['ts'] >= window_start) & (df_metric_sub['ts'] <= window_end)].copy()

# Use list for selecting multiple columns
metric_window_stats = (
    df_metric_win_sub
    .groupby(['cmdb_id','kpi_name'], dropna=False)[['value','ts']]
    .agg(points_in_window=('value','count'),
         max_value_in_window=('value','max'))
    .reset_index()
)

# Merge baseline with window stats
merged = metric_baseline.merge(metric_window_stats, on=['cmdb_id','kpi_name'], how='left')

# 3) Mark anomalies and compute anomaly-related metrics including longest consecutive run
if not df_metric_win_sub.empty:
    mrows = df_metric_win_sub.merge(metric_baseline, on=['cmdb_id','kpi_name'], how='left')
    mrows['is_anom'] = (mrows['value'] >= mrows['global_p95']).astype(int)
    # anomaly counts and earliest anomaly timestamp
    anomaly_counts = (
        mrows[mrows['is_anom']==1]
        .groupby(['cmdb_id','kpi_name'], dropna=False)
        .agg(anomaly_count_in_window=('is_anom','sum'),
             earliest_anomaly_ts=('ts','min'))
        .reset_index()
    )
    # compute longest consecutive anomaly run per pair (consecutive minutes where is_anom==1)
    def longest_run_minutes(ts_series):
        if ts_series.empty:
            return 0
        s = ts_series.sort_values().reset_index(drop=True)
        if len(s) == 0:
            return 0
        diffs = s.diff().dt.total_seconds().fillna(0)
        run_id = (diffs != 60).cumsum()
        run_lengths = s.groupby(run_id).size()
        return int(run_lengths.max())
    anom_runs = (
        mrows[mrows['is_anom']==1]
        .groupby(['cmdb_id','kpi_name'], dropna=False)['ts']
        .apply(longest_run_minutes)
        .reset_index(name='longest_consecutive_anomaly_length_in_minutes')
    )
    # combine anomaly_counts and runs
    anomaly_meta = anomaly_counts.merge(anom_runs, on=['cmdb_id','kpi_name'], how='left')
    if not anomaly_meta.empty:
        anomaly_meta['earliest_anomaly_timestamp_iso_utc'] = anomaly_meta['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').str.replace('+0000','+00:00', regex=False)
        anomaly_meta = anomaly_meta.drop(columns=['earliest_anomaly_ts'])
else:
    anomaly_meta = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count_in_window','longest_consecutive_anomaly_length_in_minutes','earliest_anomaly_timestamp_iso_utc'])

# Merge anomaly meta into merged table
final = merged.merge(anomaly_meta, on=['cmdb_id','kpi_name'], how='left')

# Fill defaults for missing window data
final['points_in_window'] = final['points_in_window'].fillna(0).astype(int)
final['anomaly_count_in_window'] = final['anomaly_count_in_window'].fillna(0).astype(int)
final['longest_consecutive_anomaly_length_in_minutes'] = final['longest_consecutive_anomaly_length_in_minutes'].fillna(0).astype(int)
final['earliest_anomaly_timestamp_iso_utc'] = final['earliest_anomaly_timestamp_iso_utc'].where(final['earliest_anomaly_timestamp_iso_utc'].notna(), None)
final['max_value_in_window'] = final['max_value_in_window'].where(final['max_value_in_window'].notna(), None)

# 4) severity_ratio
def compute_severity(row):
    if pd.isna(row['max_value_in_window']):
        return None
    return float(row['max_value_in_window']) / (float(row['global_p95']) + 1e-9)

final['severity_ratio'] = final.apply(compute_severity, axis=1)

# 5) Select and sort results, limit to 50 rows
focused_result = final[[
    'cmdb_id','kpi_name','global_p95','total_points','points_in_window',
    'anomaly_count_in_window','longest_consecutive_anomaly_length_in_minutes',
    'earliest_anomaly_timestamp_iso_utc','max_value_in_window','severity_ratio'
]].sort_values(by=['anomaly_count_in_window','severity_ratio'], ascending=[False, False]).reset_index(drop=True)

focused_result_top = focused_result.head(50)

# Per-component KPI lists and counts
component_kpis = (
    metric_baseline
    .groupby('cmdb_id', dropna=False)
    .agg(kpi_list=('kpi_name', lambda s: sorted(s.unique())),
         kpi_count=('kpi_name', lambda s: int(s.nunique())))
    .reset_index()
)

# Return compact results (DataFrames)
focused_result_top, component_kpis

```
Out[3]:
```
Focused summary for cmdb_ids ["carts","catalogue","orders","payment","user"]
(incident window: 2024-01-20 03:50:34 → 2024-01-20 04:20:34 UTC)

Per-component KPI coverage
- carts (7 KPIs): cpu, diskio, latency-50, latency-90, mem, socket, workload
- catalogue (6 KPIs): cpu, latency-50, latency-90, mem, socket, workload
- orders (7 KPIs): cpu, diskio, latency-50, latency-90, mem, socket, workload
- payment (6 KPIs): cpu, latency-50, latency-90, mem, socket, workload
- user (6 KPIs): cpu, latency-50, latency-90, mem, socket, workload

Key findings (compact):
- Widespread socket anomalies:
  - catalogue / socket: global P95 = 7.0, total_points = 25, points_in_window = 25, anomaly_count = 25, longest consecutive anomalous run = 25 minutes, earliest anomaly ≈ 2024-01-20T03:53:00Z, max = 7.0, severity_ratio = 1.0
  - payment / socket: global P95 = 6.0, total_points = 25, points_in_window = 25, anomaly_count = 25, longest run = 25 minutes, earliest ≈ 2024-01-20T03:53:00Z, severity_ratio = 1.0
  - user / socket: global P95 = 20.0, total_points = 25, points_in_window = 25, anomaly_count = 24, longest run = 13 minutes, earliest ≈ 2024-01-20T03:53:00Z, severity_ratio = 1.0
  Interpretation: socket KPI is above its historical P95 for essentially the whole window for multiple services — indicates a sustained socket/network/socket-layer issue affecting catalogue, payment, and user.

- Disk I/O / spike in carts:
  - carts / diskio: global P95 ≈ 51,104, total_points = 25, points_in_window = 25, anomaly_count = 2, longest run = 2 minutes, earliest anomaly ≈ 2024-01-20T03:53:00Z, max ≈ 151,477, severity_ratio ≈ 2.96
  Interpretation: carts shows a short but large disk I/O spike (severity_ratio ~3), suggesting brief heavy disk activity on carts.

- Orders diskio is degenerate:
  - orders / diskio: global P95 = 0.0, total_points = 25, points_in_window = 25, anomaly_count = 25, max = 0, severity_ratio = 0
  Interpretation: diskio baseline is zero, so counts appear anomalous by the >=P95 rule but carry no practical signal (max is also 0).

- Memory / CPU / latency KPIs:
  - Several mem and cpu KPIs show very small anomaly counts (typically 1–2 points) with severity ratios ≈1.00–1.003 — minor elevations, not sustained.
  - Example: carts/user/payment mem and cpu show small, short anomalies (earliest times in the window between ~03:54–04:16).

Overall conclusion (concise)
- The dominant, sustained signal across these five services is socket-related: catalogue.socket and payment.socket are above their historical P95 for the entire window, and user.socket is nearly continuous (24/25 points, 13-minute longest run). This points to a socket/network-level issue impacting multiple services during the incident window.
- Separately, carts experienced short but significant disk I/O spikes (high severity ratio, short runs) that may be a local resource issue on carts.
- Other KPI anomalies (mem/cpu/latency) are minor and short-lived.

If you want, I can return the full compact table (up to 50 rows) shown previously as CSV-like text for further inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name    global_p95  total_points  points_in_window  anomaly_count_in_window  longest_consecutive_anomaly_length_in_minutes earliest_anomaly_timestamp_iso_utc  max_value_in_window  severity_ratio
0   catalogue      socket  7.000000e+00            25                25                       25                                             25          2024-01-20T03:53:00+00:00         7.000000e+00        1.000000
1     payment      socket  6.000000e+00            25                25                       25                                             25          2024-01-20T03:53:00+00:00         6.000000e+00        1.000000
2      orders      diskio  0.000000e+00            25                25                       25                                             25          2024-01-20T03:53:00+00:00         0.000000e+00        0.000000
3        user      socket  2.000000e+01            25                25                       24                                             13          2024-01-20T03:53:00+00:00         2.000000e+01        1.000000
4       carts      diskio  5.110429e+04            25                25                        2                                              2          2024-01-20T03:53:00+00:00         1.514773e+05        2.964083
..        ...         ...           ...           ...               ...                      ...                                            ...                                ...                  ...             ...
27       user  latency-50  5.340827e-03            25                25                        2                                              1          2024-01-20T04:01:00+00:00         5.356341e-03        1.002905
28       user         mem  8.510737e+06            25                25                        2                                              1          2024-01-20T04:14:00+00:00         8.530096e+06        1.002275
29      carts         mem  2.279152e+08            25                25                        2                                              1          2024-01-20T04:10:00+00:00         2.282720e+08        1.001565
30      carts         cpu  7.901453e+01            25                25                        2                                              1          2024-01-20T04:07:00+00:00         7.908008e+01        1.000830
31    payment         mem  4.619674e+06            25                25                        2                                              2          2024-01-20T04:16:00+00:00         4.621039e+06        1.000296

[32 rows x 10 columns],      cmdb_id                                           kpi_list  kpi_count
0      carts  [cpu, diskio, latency-50, latency-90, mem, soc...          7
1  catalogue  [cpu, latency-50, latency-90, mem, socket, wor...          6
2     orders  [cpu, diskio, latency-50, latency-90, mem, soc...          7
3    payment  [cpu, latency-50, latency-90, mem, socket, wor...          6
4       user  [cpu, latency-50, latency-90, mem, socket, wor...          6)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```